In [ ]:
# ==========================================================
# INSTALAÇÃO DAS BIBLIOTECAS
# ==========================================================
!pip -q install opencv-python-headless scikit-image matplotlib pandas numpy tqdm deepface lpips torch torchvision

# ==========================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lpips

from tqdm import tqdm
from google.colab import files
from skimage.metrics import structural_similarity as ssim
from deepface import DeepFace

In [ ]:
# ==========================================================
# CONFIGURAÇÕES
# ==========================================================
MAX_FRAMES = 60
FACE_SIZE = (224, 224)
DETECTOR_BACKEND = "opencv"   # pode tentar: "retinaface", "mtcnn", "opencv"
EMBEDDING_MODEL = "Facenet512"

# Pesos do score final
WEIGHTS = {
    "identity": 0.40,
    "lpips": 0.20,
    "ssim": 0.15,
    "temporal_identity": 0.15,
    "temporal_visual": 0.10
}

# ==========================================================
# MODELO LPIPS
# ==========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
lpips_model = lpips.LPIPS(net='alex').to(device)

In [ ]:
# ==========================================================
# LIMPEZA OPCIONAL
# ==========================================================
print("=== LIMPEZA OPCIONAL ===")
limpar = input("Deseja limpar arquivos antigos antes do novo upload? (s/n): ").strip().lower()

if limpar == "s":
    removidos = []
    for f in os.listdir():
        nome = f.lower()
        if (
            nome.startswith("video_original")
            or nome.startswith("imagem_original")
            or nome.startswith("teste_")
            or nome.endswith(".csv")
        ):
            try:
                os.remove(f)
                removidos.append(f)
            except Exception as e:
                print(f"Não foi possível remover {f}: {e}")

    plt.close("all")
    print("Arquivos removidos:", removidos if removidos else "nenhum")

In [ ]:
# ==========================================================
# UPLOAD DOS ARQUIVOS
# ==========================================================

print("=" * 50)
print("      FAÇA O UPLOAD DOS ARQUIVOS")
print("=" * 50)

print("""
Para que o material seja lido corretamente, carregue os arquivos já com os nomes padronizados conforme abaixo abaixo:

[1] Vídeo original (Alvo)
    Nome: video_original. (".mp4", ".mov", ".avi", ".mkv", ".webm")

[2] Imagem original (Origem)
    Nome: imagem_original.jpg (".png", ".jpg", ".jpeg", ".webp", ".bmp")

[3] Vídeos de teste (Resultante)
    Nome: Teste_n (".mp4", ".mov", ".avi", ".mkv", ".webm")
        - n = quantidade de videos desejadas, exemplo: Teste_1, Teste_2, Teste_3, etc.
        - Obs: A quantidade de videos será reconhecida automaticamente desde que mantenha o padrão inicial de Descrição "Teste_"

[NOTA]:Pode ser carregado quantos vídeos testes desejar, desde que, todos sejam para comparativo com o video original e origem e mantenham o padrão de descrição)

""")


uploaded = files.upload()

In [ ]:
print("=" * 50)
print("      FAÇA O UPLOAD DOS ARQUIVOS")
print("=" * 50)

print("""
Para que o material seja lido corretamente, carregue os arquivos já com os nomes padronizados conforme abaixo abaixo:

[1] Vídeo original (Alvo)
    Nome: video_original. (".mp4", ".mov", ".avi", ".mkv", ".webm")

[2] Imagem original (Origem)
    Nome: imagem_original.jpg (".png", ".jpg", ".jpeg", ".webp", ".bmp")

[3] Vídeos de teste (Resultante)
    Nome: Teste_n (".mp4", ".mov", ".avi", ".mkv", ".webm")
        - n = quantidade de videos desejadas, exemplo: Teste_1, Teste_2, Teste_3, etc.
        - Obs: A quantidade de videos será reconhecida automaticamente desde que mantenha o padrão inicial de Descrição "Teste_"

[NOTA]:Pode ser carregado quantos vídeos testes desejar, desde que, todos sejam para comparativo com o video original e origem e mantenham o padrão de descrição)

""")

In [ ]:
# ==========================================================
# DETECÇÃO AUTOMÁTICA DOS ARQUIVOS
# ==========================================================
print("\n=== DETECÇÃO AUTOMÁTICA ===")
arquivos = os.listdir()

VIDEO_ORIGINAL = None
IMAGEM_ORIGINAL = None
VIDEOS_TESTE = []

for f in arquivos:
    nome = f.lower()

    if nome.startswith("video_original") and nome.endswith((".mp4", ".mov", ".avi", ".mkv", ".webm")):
        VIDEO_ORIGINAL = f

    elif nome.startswith("imagem_original") and nome.endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp")):
        IMAGEM_ORIGINAL = f

    elif nome.startswith("teste_") and nome.endswith((".mp4", ".mov", ".avi", ".mkv", ".webm")):
        VIDEOS_TESTE.append(f)

VIDEOS_TESTE = sorted(VIDEOS_TESTE)

print("Vídeo original detectado:", VIDEO_ORIGINAL)
print("Imagem original detectada:", IMAGEM_ORIGINAL)
print("Vídeos de teste detectados:", VIDEOS_TESTE)

if VIDEO_ORIGINAL is None:
    raise FileNotFoundError("Nenhum arquivo 'video_original' foi encontrado.")

if IMAGEM_ORIGINAL is None:
    raise FileNotFoundError("Nenhum arquivo 'imagem_original' foi encontrado.")

if len(VIDEOS_TESTE) == 0:
    raise FileNotFoundError("Nenhum vídeo 'teste_' foi encontrado.")

In [ ]:
# ==========================================================
# FUNÇÕES AUXILIARES
# ==========================================================
def extract_sampled_frames(video_path, max_frames=60):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Não foi possível abrir o vídeo: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        raise ValueError(f"Vídeo sem frames válidos: {video_path}")

    sample_indices = np.linspace(0, total_frames - 1, min(max_frames, total_frames), dtype=int)

    frames = []
    current_idx = 0
    target_set = set(sample_indices.tolist())

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if current_idx in target_set:
            frames.append((current_idx, frame.copy()))

        current_idx += 1

    cap.release()
    return frames


def resize_to_same(img1, img2):
    h = min(img1.shape[0], img2.shape[0])
    w = min(img1.shape[1], img2.shape[1])
    img1_r = cv2.resize(img1, (w, h))
    img2_r = cv2.resize(img2, (w, h))
    return img1_r, img2_r


def compute_ssim_frame(img1, img2):
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    return float(ssim(gray1, gray2, data_range=255))


def bgr_to_lpips_tensor(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    rgb = rgb.astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0)
    tensor = tensor * 2 - 1  # [0,1] -> [-1,1]
    return tensor.to(device)


def compute_lpips(img1, img2):
    img1, img2 = resize_to_same(img1, img2)
    t1 = bgr_to_lpips_tensor(img1)
    t2 = bgr_to_lpips_tensor(img2)
    with torch.no_grad():
        value = lpips_model(t1, t2).item()
    return float(value)


def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1, dtype=np.float32)
    vec2 = np.array(vec2, dtype=np.float32)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return np.nan

    return float(np.dot(vec1, vec2) / (norm1 * norm2))


def extract_main_face(image_bgr, target_size=(224, 224)):
    try:
        faces = DeepFace.extract_faces(
            img_path=image_bgr,
            detector_backend=DETECTOR_BACKEND,
            enforce_detection=False,
            align=True
        )
        if not faces:
            return None

        # DeepFace retorna lista de faces; pega a maior área se houver info facial_area
        best_face = None
        best_area = -1

        for item in faces:
            facial_area = item.get("facial_area", {})
            area = facial_area.get("w", 0) * facial_area.get("h", 0)
            if area > best_area:
                best_area = area
                best_face = item

        if best_face is None:
            return None

        face_rgb = (best_face["face"] * 255).astype(np.uint8)
        face_bgr = cv2.cvtColor(face_rgb, cv2.COLOR_RGB2BGR)
        face_bgr = cv2.resize(face_bgr, target_size)
        return face_bgr

    except Exception:
        return None


def get_face_embedding(image_bgr):
    try:
        reps = DeepFace.represent(
            img_path=image_bgr,
            model_name=EMBEDDING_MODEL,
            detector_backend=DETECTOR_BACKEND,
            enforce_detection=False
        )
        if isinstance(reps, list) and len(reps) > 0:
            return reps[0]["embedding"]
        return None
    except Exception:
        return None


def compute_temporal_visual_consistency(face_frames):
    if len(face_frames) < 2:
        return np.nan

    scores = []
    for i in range(len(face_frames) - 1):
        f1, f2 = resize_to_same(face_frames[i], face_frames[i + 1])
        scores.append(compute_ssim_frame(f1, f2))

    return float(np.mean(scores)) if scores else np.nan


def compute_temporal_identity_consistency(embeddings):
    valid_embeddings = [e for e in embeddings if e is not None]
    if len(valid_embeddings) < 2:
        return np.nan

    scores = []
    for i in range(len(valid_embeddings) - 1):
        sim = cosine_similarity(valid_embeddings[i], valid_embeddings[i + 1])
        if not np.isnan(sim):
            scores.append(sim)

    return float(np.mean(scores)) if scores else np.nan


def normalize_lpips(lpips_value, max_expected=1.0):
    # LPIPS menor é melhor -> converte para score 0-100
    if np.isnan(lpips_value):
        return np.nan
    value = max(0.0, min(lpips_value, max_expected))
    score = 100 * (1 - value / max_expected)
    return float(score)


def normalize_similarity(sim_value):
    if np.isnan(sim_value):
        return np.nan
    # cos sim geralmente ~ [0,1] neste cenário
    sim_value = max(0.0, min(sim_value, 1.0))
    return float(sim_value * 100)


def classify_score_100(score):
    if np.isnan(score):
        return "não calculado"
    if score >= 85:
        return "excelente"
    elif score >= 70:
        return "muito bom"
    elif score >= 55:
        return "aceitável"
    elif score >= 40:
        return "fraco"
    return "ruim"

In [ ]:
# ==========================================================
# VALIDAÇÃO DOS ARQUIVOS
# ==========================================================
required_files = [VIDEO_ORIGINAL, IMAGEM_ORIGINAL] + VIDEOS_TESTE
for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

# ==========================================================
# PREPARAÇÃO DOS DADOS FIXOS
# ==========================================================
print("\n=== PREPARAÇÃO DOS DADOS ===")
print("Extraindo frames do vídeo original...")
original_frames_master = extract_sampled_frames(VIDEO_ORIGINAL, max_frames=MAX_FRAMES)

imagem_original = cv2.imread(IMAGEM_ORIGINAL)
if imagem_original is None:
    raise ValueError(f"Não foi possível abrir a imagem {IMAGEM_ORIGINAL}")

reference_face = extract_main_face(imagem_original, target_size=FACE_SIZE)
if reference_face is None:
    raise ValueError("Não foi possível detectar face válida em 'imagem_original'.")

reference_embedding = get_face_embedding(reference_face)
if reference_embedding is None:
    raise ValueError("Não foi possível gerar embedding da imagem_original.")

In [ ]:
# ==========================================================
# PROCESSAMENTO DOS TESTES
# ==========================================================
resumo_geral = []
todos_frames = []

for video_teste in VIDEOS_TESTE:
    nome_resultado = os.path.splitext(video_teste)[0]

    print(f"\n==============================")
    print(f"Analisando: {video_teste}")
    print(f"==============================")

    final_frames_master = extract_sampled_frames(video_teste, max_frames=MAX_FRAMES)

    num_pairs = min(len(original_frames_master), len(final_frames_master))
    if num_pairs == 0:
        print(f"Vídeo sem pares válidos: {video_teste}")
        continue

    original_frames = original_frames_master[:num_pairs]
    final_frames = final_frames_master[:num_pairs]

    frame_results = []
    detected_faces = []
    detected_embeddings = []
    successful_face_detections = 0

    for i in tqdm(range(num_pairs), desc=f"Frames {nome_resultado}"):
        idx_o, frame_o = original_frames[i]
        idx_f, frame_f = final_frames[i]

        frame_o, frame_f = resize_to_same(frame_o, frame_f)

        frame_ssim = compute_ssim_frame(frame_o, frame_f)
        frame_lpips = compute_lpips(frame_o, frame_f)

        final_face = extract_main_face(frame_f, target_size=FACE_SIZE)

        identity_sim = np.nan
        face_detected = False

        if final_face is not None:
            face_detected = True
            successful_face_detections += 1
            detected_faces.append(final_face)

            final_embedding = get_face_embedding(final_face)
            if final_embedding is not None:
                detected_embeddings.append(final_embedding)
                identity_sim = cosine_similarity(reference_embedding, final_embedding)
            else:
                detected_embeddings.append(None)
        else:
            detected_embeddings.append(None)

        frame_results.append({
            "arquivo_original": VIDEO_ORIGINAL,
            "arquivo_modelo": IMAGEM_ORIGINAL,
            "arquivo_teste": video_teste,
            "resultado_nome": nome_resultado,
            "frame_index": int(idx_o),
            "SSIM": frame_ssim,
            "LPIPS": frame_lpips,
            "IdentitySimilarity": identity_sim,
            "FaceDetected": face_detected
        })

    df_frames = pd.DataFrame(frame_results)

    temporal_visual = compute_temporal_visual_consistency(detected_faces)
    temporal_identity = compute_temporal_identity_consistency(detected_embeddings)

    ssim_mean = round(df_frames["SSIM"].mean(), 4)
    lpips_mean = round(df_frames["LPIPS"].mean(), 4)
    identity_mean = round(np.nanmean(df_frames["IdentitySimilarity"]), 4) if df_frames["IdentitySimilarity"].notna().sum() > 0 else np.nan
    face_detection_rate = round((successful_face_detections / num_pairs) * 100, 2)

    temporal_visual = round(temporal_visual, 4) if not np.isnan(temporal_visual) else np.nan
    temporal_identity = round(temporal_identity, 4) if not np.isnan(temporal_identity) else np.nan

    # Normalizações para score global
    identity_score_100 = normalize_similarity(identity_mean)
    ssim_score_100 = normalize_similarity(ssim_mean)
    temporal_visual_100 = normalize_similarity(temporal_visual)
    temporal_identity_100 = normalize_similarity(temporal_identity)
    lpips_score_100 = normalize_lpips(lpips_mean)

    weighted_parts = []
    weighted_sum = 0.0

    components = {
        "identity": identity_score_100,
        "lpips": lpips_score_100,
        "ssim": ssim_score_100,
        "temporal_identity": temporal_identity_100,
        "temporal_visual": temporal_visual_100
    }

    for key, weight in WEIGHTS.items():
        value = components[key]
        if not np.isnan(value):
            weighted_sum += value * weight
            weighted_parts.append(weight)

    score_final = round(weighted_sum / sum(weighted_parts), 2) if weighted_parts else np.nan

    # Melhor frame: ranking normalizado e consistente com o BFFV
    df_rank = df_frames.copy()
    df_rank["IdentitySimilarity"] = df_rank["IdentitySimilarity"].fillna(0)
    df_rank["SSIM"] = df_rank["SSIM"].fillna(0)
    df_rank["LPIPS"] = df_rank["LPIPS"].fillna(1)

    spatial_weights = {
        "identity": WEIGHTS["identity"],
        "ssim": WEIGHTS["ssim"],
        "lpips": WEIGHTS["lpips"]
    }
    sum_spatial = sum(spatial_weights.values())

    df_rank["score_frame"] = (
        (df_rank["IdentitySimilarity"] * (spatial_weights["identity"] / sum_spatial)) +
        (df_rank["SSIM"] * (spatial_weights["ssim"] / sum_spatial)) +
        ((1 - np.clip(df_rank["LPIPS"], 0, 1)) * (spatial_weights["lpips"] / sum_spatial))
    )

    melhor_frame_row = df_rank.sort_values("score_frame", ascending=False).iloc[0]

    resumo_geral.append({
        "resultado_nome": nome_resultado,
        "arquivo_teste": video_teste,
        "Frames_avaliados": num_pairs,

        "SSIM_medio": ssim_mean,
        "LPIPS_medio": lpips_mean,
        "IdentitySimilarity_media": identity_mean,

        "TemporalVisualConsistency_media": temporal_visual,
        "TemporalIdentityConsistency_media": temporal_identity,

        "FaceDetectionRate_%": face_detection_rate,

        "Melhor_frame": int(melhor_frame_row["frame_index"]),
        "Melhor_frame_SSIM": round(float(melhor_frame_row["SSIM"]), 4),
        "Melhor_frame_LPIPS": round(float(melhor_frame_row["LPIPS"]), 4),
        "Melhor_frame_Identity": round(float(melhor_frame_row["IdentitySimilarity"]), 4),

        "Score_Identidade_%": round(identity_score_100, 2) if not np.isnan(identity_score_100) else np.nan,
        "Score_Estrutural_%": round(ssim_score_100, 2) if not np.isnan(ssim_score_100) else np.nan,
        "Score_Percepcao_%": round(lpips_score_100, 2) if not np.isnan(lpips_score_100) else np.nan,
        "Score_TemporalVisual_%": round(temporal_visual_100, 2) if not np.isnan(temporal_visual_100) else np.nan,
        "Score_TemporalIdentidade_%": round(temporal_identity_100, 2) if not np.isnan(temporal_identity_100) else np.nan,

        "Score_final": score_final,
        "Classificacao_geral": classify_score_100(score_final)
    })

    todos_frames.append(df_frames)

In [ ]:
# ==========================================================
# DATAFRAMES FINAIS
# ==========================================================
if len(resumo_geral) == 0:
    raise ValueError("Nenhum resultado válido foi processado.")

resumo_df = pd.DataFrame(resumo_geral)
frames_df = pd.concat(todos_frames, ignore_index=True)

ranking_df = resumo_df.sort_values(
    by=["Score_final", "IdentitySimilarity_media", "TemporalIdentityConsistency_media", "SSIM_medio"],
    ascending=False
).reset_index(drop=True)

ranking_df.insert(0, "Posicao", range(1, len(ranking_df) + 1))

In [ ]:
# ==========================================================
# EXIBIÇÃO
# ==========================================================
print("\n===== TABELA RESUMO GERAL =====")
display(resumo_df)

print("\n===== RANKING FINAL =====")
display(ranking_df)

print("\n===== RESULTADOS POR FRAME (amostra) =====")
display(frames_df.head(20))

In [ ]:
# ==========================================================
# SALVAR CSVs
# ==========================================================
resumo_df.to_csv("resumo_comparativo_testes.csv", index=False)
ranking_df.to_csv("ranking_final_testes.csv", index=False)
frames_df.to_csv("metricas_por_frame_testes.csv", index=False)

print("\nArquivos salvos:")
print("- resumo_comparativo_testes.csv")
print("- ranking_final_testes.csv")
print("- metricas_por_frame_testes.csv")